# 01 - Prophet Model (Per-Series Approach)

Ce notebook entraine des modeles Prophet **PAR SERIE** (store_id + product_id).

**POURQUOI CETTE APPROCHE:**
- Prophet est concu pour des series temporelles UNIVARIEES
- Agreger toutes les series en une seule detruit les patterns specifiques a chaque magasin/produit
- XGBoost (R2=0.955) utilise les features par groupe - Prophet doit faire pareil

**STRATEGIE:**
1. Entrainer un modele Prophet distinct pour chaque combinaison (store_id, product_id)
2. Utiliser des regresseurs externes (price, promo, weekend)
3. Agreger les predictions pour l'evaluation finale

**Outputs:**
- `models/artifacts/prophet_models_YYYYMMDD.pkl`
- `models/artifacts/prophet_config_YYYYMMDD.json`
- `models/metrics/prophet_metrics_YYYYMMDD.json`

In [ ]:
import sys
import json
import datetime
import warnings
from pathlib import Path
from math import sqrt
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Suppress Prophet and cmdstanpy logs
warnings.filterwarnings('ignore')
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

# Setup project path - SAME AS train.py
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# CONFIGURATION - SAME AS XGBOOST NOTEBOOK
# ============================================================
HORIZON = 14
DATE_COL = 'date'
TARGET_COL = 'value'
TODAY = datetime.datetime.now().strftime("%Y%m%d")

# Minimum samples per series to train a model
MIN_SAMPLES_PER_SERIES = 30

# Paths
ARTIFACTS_PATH = PROJECT_ROOT / "models" / "artifacts"
METRICS_PATH = PROJECT_ROOT / "models" / "metrics"
ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
METRICS_PATH.mkdir(parents=True, exist_ok=True)

# Import project config
try:
    from src.config import TRAIN_CSV, GROUP_COLS
except ImportError:
    TRAIN_CSV = PROJECT_ROOT / "data" / "interim" / "train.csv"
    GROUP_COLS = ['store_id', 'product_id']

print(f"HORIZON={HORIZON}")
print(f"GROUP_COLS={GROUP_COLS}")

In [ ]:
# Find data file - SAME LOGIC AS XGBOOST
def find_data_file():
    candidates = [
        TRAIN_CSV,
        PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv",
        PROJECT_ROOT / "data" / "processed" / "train_features.csv",
    ]
    for folder in ["interim", "processed", "raw"]:
        folder_path = PROJECT_ROOT / "data" / folder
        if folder_path.exists():
            for f in folder_path.glob("*.csv"):
                candidates.append(f)
    
    for p in candidates:
        if p and p.exists():
            return p
    raise FileNotFoundError("No data file found")

DATA_PATH = find_data_file()
print(f"Data: {DATA_PATH}")

## 1. Load and Prepare Data (SAME AS XGBOOST)

In [ ]:
# Load raw data
df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Remap columns if needed - SAME AS XGBOOST
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})

# Handle on_promo - SAME AS XGBOOST
if 'on_promo' in df.columns:
    df['on_promo'] = df['on_promo'].map(
        {True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0}
    ).fillna(0).astype(float)

# Detect group columns
available_groups = [c for c in GROUP_COLS if c in df.columns]
print(f"Group columns: {available_groups}")

# Sort by group + date
sort_cols = available_groups + [DATE_COL]
df = df.sort_values(sort_cols).reset_index(drop=True)

# Remove rows with missing target
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print(f"\nFinal shape: {df.shape}")
print(f"Date range: {df[DATE_COL].min()} to {df[DATE_COL].max()}")
print(f"Target stats: mean={df[TARGET_COL].mean():.2f}, std={df[TARGET_COL].std():.2f}")

In [ ]:
# Create series identifier - CRITICAL FOR PER-SERIES TRAINING
if len(available_groups) > 0:
    df['series_id'] = df[available_groups].astype(str).agg('_'.join, axis=1)
else:
    df['series_id'] = 'all'

n_series = df['series_id'].nunique()
print(f"Number of unique series: {n_series}")

# Check series sizes
series_sizes = df.groupby('series_id').size()
print(f"\nSeries size stats:")
print(f"  Min: {series_sizes.min()}")
print(f"  Max: {series_sizes.max()}")
print(f"  Mean: {series_sizes.mean():.1f}")
print(f"  Series with >= {MIN_SAMPLES_PER_SERIES} samples: {(series_sizes >= MIN_SAMPLES_PER_SERIES).sum()}")

## 2. Train/Test Split (Time-based) - SAME AS XGBOOST

In [ ]:
# Split by date - EXACT SAME LOGIC AS XGBOOST NOTEBOOK
unique_dates = sorted(df[DATE_COL].unique())
if HORIZON >= len(unique_dates):
    HORIZON = max(1, len(unique_dates) // 5)

cutoff_date = unique_dates[-HORIZON]
print(f"Cutoff date: {cutoff_date}")
print(f"Test period: {cutoff_date} to {unique_dates[-1]}")

train_mask = df[DATE_COL] < cutoff_date
test_mask = df[DATE_COL] >= cutoff_date

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"\nTrain: {len(df_train)} samples")
print(f"Test: {len(df_test)} samples")

## 3. Prophet Per-Series Training

In [ ]:
from prophet import Prophet

def prepare_prophet_df(df_subset, date_col, target_col):
    """Prepare DataFrame for Prophet (requires 'ds' and 'y' columns)."""
    prophet_df = pd.DataFrame()
    prophet_df['ds'] = df_subset[date_col].values
    prophet_df['y'] = df_subset[target_col].values
    prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)
    
    # Add regressors if available
    if 'price' in df_subset.columns:
        prophet_df['price'] = df_subset['price'].values
    if 'on_promo' in df_subset.columns:
        prophet_df['on_promo'] = df_subset['on_promo'].values
    
    # Add weekend flag
    prophet_df['is_weekend'] = (pd.to_datetime(prophet_df['ds']).dt.dayofweek >= 5).astype(float)
    
    return prophet_df


def train_prophet_model(train_df, regressors=None):
    """Train a single Prophet model with optimized parameters for sales data."""
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',  # Better for sales with varying magnitude
        changepoint_prior_scale=0.1,        # Moderate flexibility
        seasonality_prior_scale=10.0,       # Strong seasonality
        holidays_prior_scale=10.0,
        changepoint_range=0.85,
        n_changepoints=25,
    )
    
    # Add monthly seasonality
    model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    
    # Add regressors
    if regressors:
        for reg in regressors:
            if reg in train_df.columns:
                model.add_regressor(reg, mode='multiplicative')
    
    # Fit model
    model.fit(train_df)
    
    return model


# Detect available regressors
REGRESSORS = []
if 'price' in df.columns:
    REGRESSORS.append('price')
if 'on_promo' in df.columns:
    REGRESSORS.append('on_promo')
REGRESSORS.append('is_weekend')

print(f"Regressors to use: {REGRESSORS}")

In [ ]:
# Train one Prophet model per series
print("="*60)
print("Training Prophet models PER SERIES")
print("="*60)

prophet_models = {}
skipped_series = []
failed_series = []

series_list = df['series_id'].unique()

for series_id in tqdm(series_list, desc="Training models"):
    # Get training data for this series
    series_train = df_train[df_train['series_id'] == series_id]
    
    # Skip series with too few data points
    if len(series_train) < MIN_SAMPLES_PER_SERIES:
        skipped_series.append(series_id)
        continue
    
    # Prepare data for Prophet
    prophet_train = prepare_prophet_df(series_train, DATE_COL, TARGET_COL)
    
    # Train model
    try:
        model = train_prophet_model(prophet_train, regressors=REGRESSORS)
        prophet_models[series_id] = model
    except Exception as e:
        failed_series.append((series_id, str(e)[:50]))

print(f"\n" + "="*60)
print(f"Training Summary:")
print(f"  Successfully trained: {len(prophet_models)} models")
print(f"  Skipped (< {MIN_SAMPLES_PER_SERIES} samples): {len(skipped_series)}")
print(f"  Failed: {len(failed_series)}")
print("="*60)

## 4. Prediction and Evaluation

In [ ]:
def predict_series(model, future_df, regressors):
    """Make predictions for a single series."""
    # Prepare future dataframe with regressors
    future = future_df[['ds']].copy()
    for reg in regressors:
        if reg in future_df.columns:
            future[reg] = future_df[reg].values
    
    forecast = model.predict(future)
    return forecast['yhat'].values


# Generate predictions for TEST set
print("Generating TEST predictions...")

test_predictions = []
test_actuals = []
test_series_ids = []

# Global mean as fallback for series without models
global_mean = df_train[TARGET_COL].mean()

for series_id in tqdm(df_test['series_id'].unique(), desc="Predicting"):
    series_test = df_test[df_test['series_id'] == series_id]
    prophet_test = prepare_prophet_df(series_test, DATE_COL, TARGET_COL)
    
    if series_id in prophet_models:
        # Use trained model
        model = prophet_models[series_id]
        preds = predict_series(model, prophet_test, REGRESSORS)
    else:
        # Fallback: use mean of that series from training, or global mean
        series_train = df_train[df_train['series_id'] == series_id]
        if len(series_train) > 0:
            preds = np.full(len(series_test), series_train[TARGET_COL].mean())
        else:
            preds = np.full(len(series_test), global_mean)
    
    # Clip negative predictions
    preds = np.clip(preds, 0, None)
    
    test_predictions.extend(preds)
    test_actuals.extend(prophet_test['y'].values)
    test_series_ids.extend([series_id] * len(preds))

# Convert to arrays
y_test = np.array(test_actuals)
y_pred_test = np.array(test_predictions)

print(f"\nTest predictions: {len(y_pred_test)}")
print(f"Prediction range: [{y_pred_test.min():.2f}, {y_pred_test.max():.2f}]")
print(f"Actual range: [{y_test.min():.2f}, {y_test.max():.2f}]")

In [ ]:
# Generate predictions for TRAIN set (for comparison with XGBoost)
print("Generating TRAIN predictions...")

train_predictions = []
train_actuals = []

for series_id in tqdm(df_train['series_id'].unique(), desc="Predicting train"):
    series_train = df_train[df_train['series_id'] == series_id]
    prophet_train = prepare_prophet_df(series_train, DATE_COL, TARGET_COL)
    
    if series_id in prophet_models:
        model = prophet_models[series_id]
        preds = predict_series(model, prophet_train, REGRESSORS)
    else:
        preds = np.full(len(series_train), series_train[TARGET_COL].mean())
    
    preds = np.clip(preds, 0, None)
    train_predictions.extend(preds)
    train_actuals.extend(prophet_train['y'].values)

y_train = np.array(train_actuals)
y_pred_train = np.array(train_predictions)

In [ ]:
# Calculate all metrics - SAME FORMAT AS XGBOOST
def compute_all_metrics(y_true, y_pred, prefix=''):
    """Compute all metrics in the same format as XGBoost notebook."""
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    bias = float(np.mean(y_pred - y_true))
    
    # Safe MAPE
    mask = np.abs(y_true) > 1.0
    mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.sum() > 0 else None
    
    # sMAPE
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape = float(np.mean(np.abs(y_true - y_pred) / denom_safe) * 100)
    
    return {
        f'MAE_{prefix}': mae,
        f'RMSE_{prefix}': rmse,
        f'R2_{prefix}': r2,
        f'Bias_{prefix}': bias,
        f'MAPE_{prefix}': mape,
        f'sMAPE_{prefix}': smape,
    }

# Train metrics
train_metrics = compute_all_metrics(y_train, y_pred_train, 'train')

# Test metrics
test_metrics = compute_all_metrics(y_test, y_pred_test, 'test')

# Bias-corrected test metrics
bias_test = test_metrics['Bias_test']
y_pred_test_corrected = np.clip(y_pred_test - bias_test, 0, None)
corrected_metrics = compute_all_metrics(y_test, y_pred_test_corrected, 'test_corrected')

# Overfit ratio
overfit_ratio = test_metrics['RMSE_test'] / train_metrics['RMSE_train'] if train_metrics['RMSE_train'] > 0 else float('inf')

print("\n" + "="*60)
print("[Prophet Per-Series] RESULTS")
print("="*60)
print(f"  Train  -> MAE={train_metrics['MAE_train']:.4f}  RMSE={train_metrics['RMSE_train']:.4f}  R2={train_metrics['R2_train']:.4f}")
print(f"  Test   -> MAE={test_metrics['MAE_test']:.4f}  RMSE={test_metrics['RMSE_test']:.4f}  R2={test_metrics['R2_test']:.4f}")
print(f"  Test*  -> MAE={corrected_metrics['MAE_test_corrected']:.4f}  RMSE={corrected_metrics['RMSE_test_corrected']:.4f}  R2={corrected_metrics['R2_test_corrected']:.4f}  (* bias-corrected)")
print(f"  Bias   -> train={train_metrics['Bias_train']:.4f}  test={bias_test:.4f}")
print(f"  MAPE   -> {test_metrics['MAPE_test']:.2f}%" if test_metrics['MAPE_test'] else "  MAPE   -> N/A")
print(f"  sMAPE  -> {test_metrics['sMAPE_test']:.2f}%")
print(f"  Overfit ratio (RMSE_test/RMSE_train): {overfit_ratio:.2f}")
print("="*60)

## 5. Save Artifacts (SAME FORMAT AS XGBOOST)

In [ ]:
# Save models
model_path = ARTIFACTS_PATH / f"prophet_models_{TODAY}.pkl"
joblib.dump(prophet_models, model_path)
print(f"Models saved to: {model_path}")

# Save configuration
config = {
    'model_type': 'prophet_per_series',
    'n_series_total': len(series_list),
    'n_models_trained': len(prophet_models),
    'n_skipped': len(skipped_series),
    'n_failed': len(failed_series),
    'min_samples_per_series': MIN_SAMPLES_PER_SERIES,
    'horizon': HORIZON,
    'regressors': REGRESSORS,
    'prophet_params': {
        'yearly_seasonality': True,
        'weekly_seasonality': True,
        'seasonality_mode': 'multiplicative',
        'changepoint_prior_scale': 0.1,
        'seasonality_prior_scale': 10.0,
        'changepoint_range': 0.85,
        'n_changepoints': 25,
        'monthly_seasonality': True,
    },
    'created_at': datetime.datetime.now().isoformat(),
    'data_path': str(DATA_PATH),
    'train_samples': len(df_train),
    'test_samples': len(df_test),
}

config_path = ARTIFACTS_PATH / f"prophet_config_{TODAY}.json"
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f"Config saved to: {config_path}")

# Save metrics (SAME FORMAT AS XGBOOST for evaluate.py compatibility)
metrics = {
    'MAE_train': train_metrics['MAE_train'],
    'RMSE_train': train_metrics['RMSE_train'],
    'R2_train': train_metrics['R2_train'],
    'Bias_train': train_metrics['Bias_train'],
    'MAE_test': test_metrics['MAE_test'],
    'RMSE_test': test_metrics['RMSE_test'],
    'R2_test': test_metrics['R2_test'],
    'Bias_test': test_metrics['Bias_test'],
    'MAE_test_corrected': corrected_metrics['MAE_test_corrected'],
    'RMSE_test_corrected': corrected_metrics['RMSE_test_corrected'],
    'R2_test_corrected': corrected_metrics['R2_test_corrected'],
    'MAPE_test': test_metrics['MAPE_test'],
    'sMAPE_test': test_metrics['sMAPE_test'],
    'overfit_ratio': overfit_ratio,
    'n_models': len(prophet_models),
}

metrics_path = METRICS_PATH / f"prophet_metrics_{TODAY}.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

## 6. Visualization

In [ ]:
# Plot predictions vs actuals - SAME STYLE AS XGBOOST
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Scatter plot (test)
ax = axes[0, 0]
ax.scatter(y_test, y_pred_test, alpha=0.5, s=10)
max_val = max(y_test.max(), y_pred_test.max())
ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect', linewidth=2)
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title(f'Prophet Per-Series: Test R2={test_metrics["R2_test"]:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Residuals distribution
ax = axes[0, 1]
residuals = y_pred_test - y_test
ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
ax.axvline(x=0, color='r', linestyle='--', linewidth=2)
ax.axvline(x=bias_test, color='orange', linestyle=':', linewidth=2, label=f'Bias={bias_test:.2f}')
ax.set_xlabel('Residual (Pred - Actual)')
ax.set_ylabel('Frequency')
ax.set_title('Test Residuals Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Time series for a sample series
ax = axes[1, 0]
if len(prophet_models) > 0:
    sample_series = list(prophet_models.keys())[0]
    series_data = df[df['series_id'] == sample_series].copy()
    series_train_data = series_data[series_data[DATE_COL] < cutoff_date]
    series_test_data = series_data[series_data[DATE_COL] >= cutoff_date]
    
    ax.plot(series_train_data[DATE_COL], series_train_data[TARGET_COL], 
            'b-', label='Train', alpha=0.7, linewidth=1)
    ax.plot(series_test_data[DATE_COL], series_test_data[TARGET_COL], 
            'g-', label='Test Actual', alpha=0.9, linewidth=2)
    
    # Get predictions for this series
    prophet_test = prepare_prophet_df(series_test_data, DATE_COL, TARGET_COL)
    model = prophet_models[sample_series]
    preds = predict_series(model, prophet_test, REGRESSORS)
    ax.plot(series_test_data[DATE_COL], np.clip(preds, 0, None), 
            'r--', label='Prophet Pred', linewidth=2)
    
    ax.axvline(x=cutoff_date, color='gray', linestyle=':', label='Cutoff')
    ax.set_title(f'Sample Series: {sample_series}')
    ax.legend(loc='upper left', fontsize=8)
else:
    ax.text(0.5, 0.5, 'No models trained', ha='center', va='center', transform=ax.transAxes)
ax.set_xlabel('Date')
ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)

# 4. Prediction error by series
ax = axes[1, 1]
# Calculate MAE per series
series_errors = []
for sid in df_test['series_id'].unique():
    mask = np.array(test_series_ids) == sid
    if mask.sum() > 0:
        mae_s = mean_absolute_error(y_test[mask], y_pred_test[mask])
        series_errors.append(mae_s)

ax.hist(series_errors, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(x=np.mean(series_errors), color='r', linestyle='--', 
           label=f'Mean MAE={np.mean(series_errors):.2f}')
ax.set_xlabel('MAE per Series')
ax.set_ylabel('Number of Series')
ax.set_title('Distribution of MAE Across Series')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(METRICS_PATH / f"prophet_plots_{TODAY}.png", dpi=150)
plt.show()

print(f"\nPlot saved to: {METRICS_PATH / f'prophet_plots_{TODAY}.png'}")

In [ ]:
# Show Prophet components for a sample series
if len(prophet_models) > 0:
    sample_series = list(prophet_models.keys())[0]
    model = prophet_models[sample_series]
    
    series_data = df[df['series_id'] == sample_series]
    prophet_all = prepare_prophet_df(series_data, DATE_COL, TARGET_COL)
    
    # Create future with regressors
    future_all = prophet_all[['ds']].copy()
    for reg in REGRESSORS:
        if reg in prophet_all.columns:
            future_all[reg] = prophet_all[reg].values
    
    forecast_all = model.predict(future_all)
    
    fig = model.plot_components(forecast_all)
    plt.suptitle(f'Prophet Components: {sample_series}', y=1.02)
    plt.tight_layout()
    plt.savefig(METRICS_PATH / f"prophet_components_{TODAY}.png", dpi=150)
    plt.show()
else:
    print("No models to visualize components")

## 7. Summary

In [ ]:
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\nModel: Prophet (Per-Series)")
print(f"Number of individual models: {len(prophet_models)} / {len(series_list)} series")
print(f"\nPerformance:")
print(f"  Train -> MAE={train_metrics['MAE_train']:.2f}  RMSE={train_metrics['RMSE_train']:.2f}  R2={train_metrics['R2_train']:.4f}")
print(f"  Test  -> MAE={test_metrics['MAE_test']:.2f}  RMSE={test_metrics['RMSE_test']:.2f}  R2={test_metrics['R2_test']:.4f}")
print(f"\nComparison with XGBoost (target):")
print(f"  XGBoost -> R2=0.955, MAE=61.97, RMSE=109.20")
print(f"  Prophet -> R2={test_metrics['R2_test']:.3f}, MAE={test_metrics['MAE_test']:.2f}, RMSE={test_metrics['RMSE_test']:.2f}")
print(f"\nArtifacts:")
print(f"  - {model_path}")
print(f"  - {config_path}")
print(f"  - {metrics_path}")
print("="*60)